# Standard Benchmark Suite (Lite)

This notebook adds a small, budget-aware benchmark harness for `AutoFillGraph v5`.

Default benchmark plan:
- `FUNSD` (official original form benchmark, full download)
- `XFUND` single-language sampled benchmark (official release assets, capped docs)

Design constraints:
- hard on-disk dataset cap of **2 GB total** across all downloaded benchmark data
- reuse the `Prototype5` core agent instead of forking logic
- evaluate only the task that matches this repo: **memory-grounded key-value autofill**

What is scored:
- raw field-label to property mapping accuracy
- exact-fill accuracy after learning from benchmark key-value pairs
- unsupported-field abstention accuracy

Notes:
- `FUNSD` is a direct fit for noisy scanned forms.
- `XFUND` is a stronger standard benchmark for multilingual form understanding, but this notebook keeps it lightweight by downloading only one language and pruning to a small doc cap.
- Full `CORD v2` is intentionally not enabled by default because the official Hugging Face dataset card lists **2.31 GB** total file size, which would consume the full budget by itself.


In [ ]:
from __future__ import annotations

import json
import os
import re
import zipfile
from collections import OrderedDict
from pathlib import Path
from typing import Dict, Iterable, List, Optional

import numpy as np
import pandas as pd
import requests
from sentence_transformers import SentenceTransformer

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except Exception:
    plt = None
    HAS_MPL = False

ROOT = Path.cwd()
CORE_NOTEBOOK = ROOT / "Baseline" / "Prototype5.ipynb"
BENCH_ROOT = ROOT / "data" / "standard_benchmarks_lite"
BENCH_ROOT.mkdir(parents=True, exist_ok=True)

MAX_DATASET_BYTES = 2 * 1024**3  # 2 GB hard cap across all local benchmark data

RUN_FUNSD = True
RUN_XFUND = True
XFUND_LANG = "de"  # use a single language to stay well under budget
XFUND_MAX_DOCS = {"train": 40, "val": 20}

BASE_EMBEDDER_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
MULTILINGUAL_EMBEDDER_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

FUNSD_URL = "https://guillaumejaume.github.io/FUNSD/dataset.zip"
XFUND_BASE_URL = "https://github.com/doc-analysis/XFUND/releases/download/v1.0/"

CANONICAL_PROMPTS = {
    "full_name": "Full Name",
    "email": "Email",
    "work_email": "Work Email",
    "phone": "Phone Number",
    "address": "Current Address",
    "city": "City",
    "state": "State",
    "zip_code": "ZIP Code",
    "country": "Country",
    "citizenship": "Citizenship",
    "passport_number": "Passport Number",
    "visa_status": "Visa Status",
    "university": "University",
    "degree": "Degree Program",
    "department": "Academic Department",
    "gpa": "GPA",
    "advisor": "Advisor",
    "graduation_date": "Graduation Date",
    "employer": "Current Employer",
    "job_title": "Job Title",
    "bank_name": "Bank Name",
    "tax_id": "Tax ID",
}

QUESTION_PATTERNS = OrderedDict([
    ("full_name", [r"\bfull\s*name\b", r"\bname\b", r"applicant", r"customer"]),
    ("email", [r"e-?mail", r"mail address"]),
    ("work_email", [r"institutional e-?mail", r"work e-?mail", r"official correspondence"]),
    ("phone", [r"phone", r"mobile", r"telephone", r"tel\.?", r"cell"]),
    ("address", [r"address", r"street"]),
    ("city", [r"\bcity\b", r"town"]),
    ("state", [r"\bstate\b", r"province", r"region"]),
    ("zip_code", [r"zip", r"postal"]),
    ("country", [r"country of residence", r"country"]),
    ("citizenship", [r"citizenship", r"nationality"]),
    ("passport_number", [r"passport"]),
    ("visa_status", [r"visa", r"immigration status"]),
    ("university", [r"university", r"college", r"school", r"institution"]),
    ("degree", [r"degree", r"program", r"major"]),
    ("department", [r"department", r"faculty"]),
    ("gpa", [r"\bgpa\b", r"grade", r"academic score", r"academic performance"]),
    ("advisor", [r"advisor", r"supervisor", r"mentor"]),
    ("graduation_date", [r"graduation", r"date of completion", r"completion date"]),
    ("employer", [r"employer", r"company", r"organization"]),
    ("job_title", [r"job title", r"position", r"role"]),
    ("bank_name", [r"bank name", r"bank"]),
    ("tax_id", [r"tax id", r"tin", r"ssn"]),
])


def norm_text(value: str) -> str:
    text = str(value or "").lower()
    text = re.sub(r"[^a-z0-9@.+/\-]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def human_bytes(n: int) -> str:
    units = ["B", "KB", "MB", "GB", "TB"]
    size = float(n)
    for unit in units:
        if size < 1024 or unit == units[-1]:
            return f"{size:.1f} {unit}"
        size /= 1024.0
    return f"{n} B"


def dir_size_bytes(path: Path) -> int:
    if not path.exists():
        return 0
    total = 0
    for item in path.rglob("*"):
        if item.is_file():
            total += item.stat().st_size
    return total


def assert_under_budget(root: Path = BENCH_ROOT, cap: int = MAX_DATASET_BYTES) -> int:
    total = dir_size_bytes(root)
    if total > cap:
        raise RuntimeError(
            f"Dataset budget exceeded: {human_bytes(total)} > {human_bytes(cap)}. "
            "Reduce benchmark sample sizes before proceeding."
        )
    return total


def exactish_match(pred, gold) -> bool:
    p = norm_text(pred)
    g = norm_text(gold)
    if not p or not g:
        return False
    if p == g:
        return True
    p_compact = re.sub(r"[^a-z0-9]+", "", p)
    g_compact = re.sub(r"[^a-z0-9]+", "", g)
    return p_compact == g_compact


print(f"Core notebook: {CORE_NOTEBOOK}")
print(f"Benchmark root: {BENCH_ROOT}")
print(f"Hard dataset cap: {human_bytes(MAX_DATASET_BYTES)}")
print(f"Current local benchmark usage: {human_bytes(assert_under_budget())}")


In [ ]:
def load_prototype5_core(nb_path: Path, cell_ids: Iterable[str]) -> None:
    # Load selected definition cells from Prototype5 without eager model/API probes.
    payload = json.loads(nb_path.read_text(encoding="utf-8"))
    wanted = set(cell_ids)
    found = {}
    for cell in payload["cells"]:
        if cell.get("id") in wanted:
            found[cell["id"]] = "".join(cell.get("source", []))

    missing = wanted.difference(found)
    if missing:
        raise KeyError(f"Missing cell ids in {nb_path.name}: {sorted(missing)}")

    def filter_source(cell_id: str, src: str) -> str:
        lines = src.splitlines()
        if cell_id == "cell-init":
            filtered = []
            for line in lines:
                if "Loading MiniLM" in line or 'SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")' in line or "MiniLM ready" in line:
                    continue
                filtered.append(line)
            src = "\n".join(filtered) + "\n"
            lines = src.splitlines()

        if any("if LLM.available()" in line for line in lines):
            filtered = []
            in_llm_probe = False
            for line in lines:
                if line.strip().startswith("if LLM.available()"):
                    in_llm_probe = True
                    continue
                if in_llm_probe:
                    if "All components defined" in line:
                        in_llm_probe = False
                        filtered.append(line)
                    continue
                filtered.append(line)
            src = "\n".join(filtered) + "\n"

        if cell_id == "cell-agent":
            marker = "\nAGENT = AutoFillAgent("
            if marker in src:
                return src.split(marker, 1)[0].rstrip() + "\n"
        return src


    for cid in ["cell-init", "cell-components", "cell-agent"]:
        exec(filter_source(cid, found[cid]), globals())


load_prototype5_core(CORE_NOTEBOOK, ["cell-init", "cell-components", "cell-agent"])

EMBEDDER_CACHE = {}


def get_embedder(model_name: str):
    if model_name not in EMBEDDER_CACHE:
        print(f"Loading embedder: {model_name}")
        EMBEDDER_CACHE[model_name] = SentenceTransformer(model_name)
    return EMBEDDER_CACHE[model_name]


def build_agent(embedder_model: str):
    # Use a no-op LLM client for benchmark consistency; benchmark scores here are local-only.
    llm = MistralClient("", DEFAULT_MODEL)
    return AutoFillAgent(get_embedder(embedder_model), llm)


print("Prototype5 core loaded without eager embedder init or live API probe.")


In [ ]:
def download_file(url: str, dest: Path) -> Path:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    print(f"Downloading: {url}")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with dest.open("wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    return dest


def unzip_file(zip_path: Path, out_dir: Path) -> Path:
    if out_dir.exists() and any(out_dir.iterdir()):
        return out_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(out_dir)
    return out_dir


def prepare_funsd(root: Path = BENCH_ROOT) -> Path:
    ds_dir = root / "funsd"
    raw_dir = ds_dir / "raw"
    zip_path = ds_dir / "dataset.zip"
    download_file(FUNSD_URL, zip_path)
    unzip_file(zip_path, raw_dir)
    total = assert_under_budget()
    print(f"FUNSD ready | local usage now: {human_bytes(total)}")
    return ds_dir


def prune_xfund_json_and_images(json_path: Path, image_root: Path, max_docs: int) -> None:
    data = json.loads(json_path.read_text(encoding="utf-8"))
    docs = data.get("documents", [])
    if len(docs) <= max_docs:
        return
    keep_docs = docs[:max_docs]
    keep_rel = {Path(doc["img"]["fname"]).as_posix() for doc in keep_docs}
    data["documents"] = keep_docs
    json_path.write_text(json.dumps(data, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    for path in list(image_root.rglob("*")):
        if not path.is_file():
            continue
        rel = path.relative_to(image_root).as_posix()
        if path.suffix.lower() in {".jpg", ".jpeg", ".png"} and rel not in keep_rel:
            path.unlink()
    for path in sorted(image_root.rglob("*"), reverse=True):
        if path.is_dir() and not any(path.iterdir()):
            path.rmdir()


def prepare_xfund_language(lang: str = XFUND_LANG, max_docs: Optional[Dict[str, int]] = None, root: Path = BENCH_ROOT):
    max_docs = max_docs or XFUND_MAX_DOCS
    ds_dir = root / "xfund" / lang
    ds_dir.mkdir(parents=True, exist_ok=True)
    prepared = {}
    for split in ["train", "val"]:
        json_path = ds_dir / f"{lang}.{split}.json"
        zip_path = ds_dir / f"{lang}.{split}.zip"
        image_dir = ds_dir / f"{split}_images"
        download_file(f"{XFUND_BASE_URL}{lang}.{split}.json", json_path)
        download_file(f"{XFUND_BASE_URL}{lang}.{split}.zip", zip_path)
        unzip_file(zip_path, image_dir)
        if max_docs.get(split):
            prune_xfund_json_and_images(json_path, image_dir, max_docs[split])
        prepared[split] = {"json": json_path, "images": image_dir}
        total = assert_under_budget()
        print(f"XFUND-{lang.upper()} {split} ready | local usage now: {human_bytes(total)}")
    return prepared


def dataset_inventory(root: Path = BENCH_ROOT) -> pd.DataFrame:
    rows = []
    if root.exists():
        for path in sorted(root.iterdir()):
            size = dir_size_bytes(path)
            rows.append({"dataset_dir": path.name, "size_bytes": size, "size_human": human_bytes(size)})
    return pd.DataFrame(rows)


In [ ]:
def extract_linked_pairs(items: List[dict]) -> List[dict]:
    by_id = {item["id"]: item for item in items if "id" in item}
    links = set()
    for item in items:
        for link in item.get("linking", []):
            if isinstance(link, list) and len(link) == 2:
                links.add(tuple(sorted(link)))
    pairs = []
    seen = set()
    for a, b in sorted(links):
        if a not in by_id or b not in by_id:
            continue
        ia, ib = by_id[a], by_id[b]
        la = str(ia.get("label", "")).lower()
        lb = str(ib.get("label", "")).lower()
        if {la, lb} != {"question", "answer"}:
            continue
        question = ia.get("text", "") if la == "question" else ib.get("text", "")
        answer = ib.get("text", "") if la == "question" else ia.get("text", "")
        question = str(question).strip()
        answer = str(answer).strip()
        if not question or not answer:
            continue
        key = (norm_text(question), norm_text(answer))
        if key in seen:
            continue
        seen.add(key)
        pairs.append({"question": question, "answer": answer})
    return pairs


def parse_funsd_annotation(annotation_path: Path) -> List[dict]:
    payload = json.loads(annotation_path.read_text(encoding="utf-8"))
    return extract_linked_pairs(payload.get("form", []))


def iter_funsd_documents(funsd_dir: Path):
    raw_root = funsd_dir / "raw" / "dataset"
    split_dirs = {
        "train": raw_root / "training_data" / "annotations",
        "test": raw_root / "testing_data" / "annotations",
    }
    for split, ann_dir in split_dirs.items():
        for ann_path in sorted(ann_dir.glob("*.json")):
            yield {
                "dataset": "FUNSD",
                "split": split,
                "doc_id": ann_path.stem,
                "pairs": parse_funsd_annotation(ann_path),
            }


def parse_xfund_json(json_path: Path) -> List[dict]:
    payload = json.loads(json_path.read_text(encoding="utf-8"))
    docs = []
    for doc in payload.get("documents", []):
        pairs = extract_linked_pairs(doc.get("document", []))
        doc_id = doc.get("id") or Path(doc.get("img", {}).get("fname", "unknown")).stem
        docs.append({"doc_id": str(doc_id), "pairs": pairs})
    return docs


def iter_xfund_documents(prepared: Dict[str, dict], lang: str):
    for split, parts in prepared.items():
        for doc in parse_xfund_json(parts["json"]):
            yield {
                "dataset": f"XFUND-{lang.upper()}",
                "split": split,
                "doc_id": doc["doc_id"],
                "pairs": doc["pairs"],
            }


def infer_expected_prop(question: str) -> Optional[str]:
    q = norm_text(question)
    for prop, patterns in QUESTION_PATTERNS.items():
        if any(re.search(pattern, q) for pattern in patterns):
            return prop
    return None


def canonical_prompt(prop: str) -> str:
    return CANONICAL_PROMPTS.get(prop, prop.replace("_", " ").title())


In [ ]:
def evaluate_document(doc: dict, embedder_model: str, max_unsupported_probes: int = 3):
    agent = build_agent(embedder_model)
    supported = OrderedDict()
    unsupported = []
    mapping_rows = []

    for pair in doc["pairs"]:
        question = pair["question"]
        answer = pair["answer"]
        expected_prop = infer_expected_prop(question)
        if expected_prop:
            predicted_prop = agent.mapper.map(question)[0]
            mapping_rows.append({
                "dataset": doc["dataset"],
                "split": doc["split"],
                "doc_id": doc["doc_id"],
                "question": question,
                "expected_prop": expected_prop,
                "predicted_prop": predicted_prop,
                "ok": predicted_prop == expected_prop,
            })
            if expected_prop not in supported:
                supported[expected_prop] = (question, answer)
        else:
            unsupported.append(question)

    if supported:
        learn_form = {raw_q: answer for raw_q, answer in supported.values()}
        agent.learn(learn_form, context="benchmark")

    fill_rows = []
    for prop, (raw_q, answer) in supported.items():
        query = canonical_prompt(prop)
        ep = agent.autofill([query], domain="general", use_llm=False)
        res = ep.results[query]
        ok = (res.prop == prop) and exactish_match(res.value, answer) and res.status != FillStatus.UNKNOWN
        fill_rows.append({
            "dataset": doc["dataset"],
            "split": doc["split"],
            "doc_id": doc["doc_id"],
            "raw_question": raw_q,
            "query": query,
            "expected_prop": prop,
            "predicted_prop": res.prop,
            "expected_value": answer,
            "predicted_value": res.value,
            "status": res.status.value if hasattr(res.status, "value") else str(res.status),
            "route": res.route.value if hasattr(res.route, "value") else str(res.route),
            "ok": ok,
        })

    abstain_rows = []
    for question in unsupported[:max_unsupported_probes]:
        ep = agent.autofill([question], domain="general", use_llm=False)
        res = ep.results[question]
        is_unknown = (res.status == FillStatus.UNKNOWN) or (str(res.value).strip().upper() == "UNKNOWN")
        abstain_rows.append({
            "dataset": doc["dataset"],
            "split": doc["split"],
            "doc_id": doc["doc_id"],
            "question": question,
            "predicted_prop": res.prop,
            "predicted_value": res.value,
            "status": res.status.value if hasattr(res.status, "value") else str(res.status),
            "ok": is_unknown,
        })

    return mapping_rows, fill_rows, abstain_rows


def run_benchmark(doc_iter, benchmark_name: str, embedder_model: str, max_docs: Optional[int] = None):
    mapping_rows = []
    fill_rows = []
    abstain_rows = []
    docs_seen = 0
    docs_with_eval = 0

    for doc in doc_iter:
        docs_seen += 1
        if max_docs is not None and docs_seen > max_docs:
            break
        m_rows, f_rows, a_rows = evaluate_document(doc, embedder_model)
        if f_rows or a_rows:
            docs_with_eval += 1
        mapping_rows.extend(m_rows)
        fill_rows.extend(f_rows)
        abstain_rows.extend(a_rows)

    mapping_df = pd.DataFrame(mapping_rows)
    fill_df = pd.DataFrame(fill_rows)
    abstain_df = pd.DataFrame(abstain_rows)

    summary = {
        "benchmark": benchmark_name,
        "embedder_model": embedder_model,
        "docs_seen": docs_seen,
        "docs_with_eval": docs_with_eval,
        "mapping_examples": len(mapping_df),
        "mapping_acc": float(mapping_df["ok"].mean()) if len(mapping_df) else float("nan"),
        "fill_examples": len(fill_df),
        "fill_acc": float(fill_df["ok"].mean()) if len(fill_df) else float("nan"),
        "abstain_examples": len(abstain_df),
        "abstain_acc": float(abstain_df["ok"].mean()) if len(abstain_df) else float("nan"),
    }

    return {
        "summary": summary,
        "mapping": mapping_df,
        "fill": fill_df,
        "abstain": abstain_df,
    }


In [ ]:
prepared = {}

if RUN_FUNSD:
    prepared["FUNSD"] = prepare_funsd()

if RUN_XFUND:
    prepared[f"XFUND-{XFUND_LANG.upper()}"] = prepare_xfund_language(XFUND_LANG, XFUND_MAX_DOCS)

inventory = dataset_inventory()
display(inventory if len(inventory) else pd.DataFrame(columns=["dataset_dir", "size_bytes", "size_human"]))
print(f"Total local benchmark usage: {human_bytes(assert_under_budget())}")


In [ ]:
results = {}

if RUN_FUNSD:
    results["FUNSD"] = run_benchmark(
        iter_funsd_documents(prepared["FUNSD"]),
        benchmark_name="FUNSD",
        embedder_model=BASE_EMBEDDER_MODEL,
    )

if RUN_XFUND:
    xfund_name = f"XFUND-{XFUND_LANG.upper()}"
    results[xfund_name] = run_benchmark(
        iter_xfund_documents(prepared[xfund_name], XFUND_LANG),
        benchmark_name=xfund_name,
        embedder_model=MULTILINGUAL_EMBEDDER_MODEL,
    )

summary_df = pd.DataFrame([pack["summary"] for pack in results.values()])
display(summary_df)

for name, pack in results.items():
    print(f"\n=== {name} ===")
    print(pack["summary"])
    if len(pack["fill"]):
        display(pack["fill"].head(10))
    if len(pack["abstain"]):
        display(pack["abstain"].head(10))


In [ ]:
if results and HAS_MPL:
    plot_df = pd.DataFrame([pack["summary"] for pack in results.values()])
    metrics = ["mapping_acc", "fill_acc", "abstain_acc"]
    fig, ax = plt.subplots(figsize=(9, 4))
    x = np.arange(len(plot_df))
    width = 0.22
    for i, metric in enumerate(metrics):
        vals = plot_df[metric].fillna(0.0).to_numpy()
        ax.bar(x + i * width, vals, width, label=metric)
    ax.set_xticks(x + width)
    ax.set_xticklabels(plot_df["benchmark"].tolist())
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Accuracy")
    ax.set_title("AutoFillGraph v5 Lite Benchmark Summary")
    ax.legend()
    plt.show()

print("\nBenchmark interpretation:")
print("- mapping_acc: raw benchmark question text -> expected AutoFillGraph property")
print("- fill_acc: exact-fill accuracy after the agent learns benchmark question/value pairs")
print("- abstain_acc: unsupported benchmark questions correctly returned as UNKNOWN")
print("\nCaveats:")
print("- This notebook benchmarks the closest comparable task for AutoFillGraph: memory-grounded key-value autofill.")
print("- It does not claim end-to-end SOTA on document understanding or web-agent planning benchmarks.")
print("- XFUND is capped to a small single-language subset to respect the 2 GB data budget.")


## Source notes

Official dataset sources used in this notebook:
- FUNSD original dataset: `https://guillaumejaume.github.io/FUNSD/dataset.zip`
- XFUND official release assets: `https://github.com/doc-analysis/XFUND/releases/tag/v1.0`

Why full CORD is not enabled by default:
- the official `naver-clova-ix/cord-v2` Hugging Face dataset card reports **2.31 GB** total file size, so downloading it in full would break the hard 2 GB benchmark-data budget for this notebook.

If you want to extend this notebook later, the clean next steps are:
1. add a streamed `CORD` negative-control section for out-of-domain abstention
2. add an optional `MiniWoB++` section for browser-side form interaction success rate
3. compare against explicit baselines on the same normalized benchmark task
